LAB 10

In [25]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("lab10").master("local[*]").getOrCreate()

df = spark.read.parquet("bigdata/silver/transactions_enriched.parquet")
df.printSchema()
df.show(5)

df.createOrReplaceTempView("silver_transactions")

spark.sql("""
    SELECT segment, ROUND(AVG(credit_score), 1) AS score_medio, COUNT(*) AS total
    FROM silver_transactions GROUP BY segment ORDER BY score_medio
""").show()

root
 |-- transaction_id: long (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- amount: float (nullable = true)
 |-- transaction_type: string (nullable = true)
 |-- status: string (nullable = true)
 |-- risk_score: float (nullable = true)
 |-- is_fraud: boolean (nullable = true)
 |-- ts: timestamp_ntz (nullable = true)
 |-- segment: string (nullable = true)
 |-- credit_score: integer (nullable = true)
 |-- year: long (nullable = true)
 |-- month: long (nullable = true)
 |-- day: long (nullable = true)
 |-- day_of_week: long (nullable = true)
 |-- amount_band: string (nullable = true)

+--------------+-----------+---------+----------------+--------+----------+--------+-------------------+-------+------------+----+-----+---+-----------+-----------+
|transaction_id|customer_id|   amount|transaction_type|  status|risk_score|is_fraud|                 ts|segment|credit_score|year|month|day|day_of_week|amount_band|
+--------------+-----------+---------+----------------+--------

In [26]:
import time
t0 = time.time()
df.filter(df.is_fraud == True).count()
print("sem cache:", time.time() - t0, "s")

df.cache()
df.count()

t0 = time.time()
df.filter(df.is_fraud == True).count()
print("com cache:", time.time() - t0, "s")

sem cache: 0.6116549968719482 s
com cache: 0.6399013996124268 s
